# Fine-tuning Phi-3 Mini on Operating Systems Q&A with QLoRA

Runs on a free Colab T4 GPU. The steps: install pinned libraries, build the 204-pair OS Q&A dataset, load Phi-3 Mini in 4-bit NF4, attach LoRA adapters (r = 16), train with TRL's `SFTTrainer`, test the model, then merge the adapters and push to the Hugging Face Hub as [Prithvi-Aithal/phi3-mini-os-finetuned](https://huggingface.co/Prithvi-Aithal/phi3-mini-os-finetuned).

In [1]:
!pip install -q transformers==4.44.0
!pip install -q peft==0.12.0
!pip install -q bitsandbytes==0.45.5
!pip install -q accelerate==0.33.0
!pip install -q datasets==2.20.0
!pip install -q trl==0.9.6

In [3]:
from datasets import Dataset

In [6]:
raw_data = [
    {
        "question": "What is an Operating System?",
        "answer": "An Operating System is a system software that acts as an intermediary between users and computer hardware. It manages resources and provides services for program execution."
    },

    {
        "question": "What are the primary goals of an Operating System?",
        "answer": "The primary goals are to execute user programs, make the system convenient to use, efficiently utilize hardware resources, and manage CPU, memory, and I/O devices."
    },

    {
        "question": "Why is an Operating System necessary?",
        "answer": "Without an operating system, application developers would have to directly interact with hardware devices, making programming complex and inefficient."
    },

    {
        "question": "What are the four components of a computer system?",
        "answer": "The four components are hardware, operating system, application programs, and users."
    },

    {
        "question": "Define kernel.",
        "answer": "The kernel is the core component of an operating system that always remains in memory and manages hardware resources and system services."
    },

    {
        "question": "What is the role of an Operating System as a resource allocator?",
        "answer": "The OS allocates CPU time, memory, and I/O devices among competing processes to ensure efficient and fair utilization."
    },

    {
        "question": "What is the role of an Operating System as a control program?",
        "answer": "It controls program execution and prevents improper use of the computer system."
    },

    {
        "question": "What is bootstrap program?",
        "answer": "Bootstrap is a firmware stored in ROM or EEPROM that initializes hardware components and loads the operating system kernel into memory."
    },

    {
        "question": "What is firmware?",
        "answer": "Firmware is software stored permanently in ROM or EEPROM that helps initialize hardware and start the operating system."
    },

    {
        "question": "What is the first user process created after booting?",
        "answer": "The init process is the first process created after booting and waits for events to occur."
    },

    {
        "question": "What is an interrupt?",
        "answer": "An interrupt is a signal that transfers control to an interrupt service routine when an event requiring attention occurs."
    },

    {
        "question": "Why is an operating system called interrupt-driven?",
        "answer": "Because the operating system responds to hardware and software events through interrupts."
    },

    {
        "question": "What is an interrupt vector?",
        "answer": "An interrupt vector is a table containing addresses of interrupt service routines."
    },

    {
        "question": "What is a trap?",
        "answer": "A trap is a software-generated interrupt caused by an error or a user request."
    },

    {
        "question": "Differentiate between hardware interrupt and software interrupt.",
        "answer": "Hardware interrupts are generated by devices like keyboards and disks, whereas software interrupts are generated by programs and exceptions."
    },

    {
        "question": "What information must be saved during interrupt handling?",
        "answer": "The operating system saves CPU registers and the program counter before servicing an interrupt."
    },

    {
        "question": "What is polling?",
        "answer": "Polling is a method where the CPU checks devices sequentially to determine which device generated an interrupt."
    },

    {
        "question": "What is a vectored interrupt system?",
        "answer": "In a vectored interrupt system, the address of the appropriate ISR is directly obtained from the interrupt vector."
    },

    {
        "question": "What is main memory?",
        "answer": "Main memory is the only large storage medium directly accessible by the CPU and is usually implemented using DRAM."
    },

    {
        "question": "What is the Von Neumann model?",
        "answer": "The Von Neumann model describes program execution as a repeated cycle of fetch, decode, and execute."
    },

    {
        "question": "Define process.",
        "answer": "A process is a program in execution along with its current activity and associated resources."
    },

    {
        "question": "What are the components of a process?",
        "answer": "A process consists of code section, data section, heap, stack, and program counter."
    },

    {
        "question": "What is a Process Control Block (PCB)?",
        "answer": "PCB is a data structure that stores information about a process such as state, program counter, registers, and scheduling information."
    },

    {
        "question": "What are the states of a process?",
        "answer": "New, Ready, Running, Waiting, and Terminated."
    },

    {
        "question": "What is process scheduling?",
        "answer": "Process scheduling determines which process gets CPU access at a given time."
    },

    {
        "question": "What is a scheduler?",
        "answer": "A scheduler is an OS component responsible for selecting processes for execution."
    },

    {
        "question": "Differentiate between long-term and short-term schedulers.",
        "answer": "Long-term schedulers decide which processes enter memory, whereas short-term schedulers select processes from the ready queue for CPU execution."
    },

    {
        "question": "What is context switching?",
        "answer": "Context switching is the process of saving the state of one process and restoring the state of another process."
    },

    {
        "question": "What is CPU burst?",
        "answer": "CPU burst is the duration for which a process executes on the CPU before waiting for I/O."
    },

    {
        "question": "What is throughput?",
        "answer": "Throughput is the number of processes completed per unit time."
    },

    {
        "question": "Define turnaround time.",
        "answer": "Turnaround time is the total time taken from process submission to completion."
    },

    {
        "question": "Define waiting time.",
        "answer": "Waiting time is the total time a process spends in the ready queue."
    },

    {
        "question": "Define response time.",
        "answer": "Response time is the interval between submission of a request and the first response."
    },

    {
        "question": "What is CPU utilization?",
        "answer": "CPU utilization is the percentage of time during which the CPU is busy executing processes."
    },

    {
        "question": "Explain FCFS scheduling.",
        "answer": "First Come First Serve scheduling executes processes in the order of arrival. It is non-preemptive and simple to implement."
    },

    {
        "question": "What is the convoy effect in FCFS scheduling?",
        "answer": "The convoy effect occurs when short processes wait behind a long process, leading to poor resource utilization."
    },

    {
        "question": "What is SJF scheduling?",
        "answer": "Shortest Job First scheduling selects the process with the smallest CPU burst time."
    },

    {
        "question": "Why is SJF considered optimal?",
        "answer": "SJF minimizes average waiting time among all scheduling algorithms."
    },

    {
        "question": "What is preemptive SJF?",
        "answer": "Preemptive SJF, also called Shortest Remaining Time First, allows interruption when a shorter job arrives."
    },

    {
        "question": "What is Priority Scheduling?",
        "answer": "Priority Scheduling assigns priorities to processes and executes the highest-priority process first."
    },

    {
        "question": "What is starvation in priority scheduling?",
        "answer": "Starvation occurs when low-priority processes wait indefinitely because higher-priority processes continuously arrive."
    },

    {
        "question": "What is aging?",
        "answer": "Aging gradually increases the priority of waiting processes to prevent starvation."
    },

    {
        "question": "Explain Round Robin scheduling.",
        "answer": "Round Robin scheduling allocates CPU to each process for a fixed time quantum and is preemptive."
    },

    {
        "question": "What is time quantum?",
        "answer": "Time quantum is the maximum CPU time allocated to a process before it is preempted."
    },

    {
        "question": "What happens if time quantum is very large?",
        "answer": "Round Robin behaves like FCFS scheduling."
    },

    {
        "question": "What happens if time quantum is very small?",
        "answer": "Too many context switches occur, increasing overhead."
    },

    {
        "question": "What is multilevel queue scheduling?",
        "answer": "Processes are permanently assigned to different queues, each with its own scheduling algorithm."
    },

    {
        "question": "What is multilevel feedback queue scheduling?",
        "answer": "Processes can move between queues depending on their behavior and CPU requirements."
    },

    {
        "question": "Differentiate preemptive and non-preemptive scheduling.",
        "answer": "Preemptive scheduling allows interruption of a running process, whereas non-preemptive scheduling allows execution until completion or blocking."
    },

    {
        "question": "Why is context switching considered overhead?",
        "answer": "Because no useful work is performed while saving and restoring process states."
    },

    {
        "question": "Which scheduling algorithm is commonly used in time-sharing systems?",
        "answer": "Round Robin scheduling is commonly used in time-sharing systems because it provides fairness and good response time."
    }
]

In [7]:
# Batch 2 (Q51–Q100): IPC, Synchronization, Semaphores, Deadlocks

raw_data.extend([

{
    "question": "What is Inter Process Communication (IPC)?",
    "answer": "IPC is a mechanism that allows processes to exchange data and synchronize their activities."
},

{
    "question": "Why is IPC required?",
    "answer": "IPC is required for information sharing, computation speedup, modularity, and convenience."
},

{
    "question": "What are the two major IPC models?",
    "answer": "The two IPC models are Shared Memory and Message Passing."
},

{
    "question": "Differentiate between independent and cooperating processes.",
    "answer": "Independent processes do not affect one another, whereas cooperating processes can affect or be affected by the execution of other processes."
},

{
    "question": "What is shared memory?",
    "answer": "Shared memory is an IPC mechanism where multiple processes access a common region of memory to communicate."
},

{
    "question": "Why is shared memory considered the fastest IPC mechanism?",
    "answer": "Because data does not need to be copied between processes and communication occurs directly through memory."
},

{
    "question": "What is message passing?",
    "answer": "Message passing is an IPC mechanism in which processes communicate by sending and receiving messages."
},

{
    "question": "Which operations are provided by message passing?",
    "answer": "The two operations are send(message) and receive(message)."
},

{
    "question": "What is the Producer-Consumer problem?",
    "answer": "The Producer-Consumer problem is a synchronization problem in which producers generate data and consumers use that data."
},

{
    "question": "What is an unbounded buffer?",
    "answer": "An unbounded buffer has no practical size limit and allows producers to continue producing items."
},

{
    "question": "What is a bounded buffer?",
    "answer": "A bounded buffer has a fixed size, causing producers to wait when the buffer is full and consumers to wait when it is empty."
},

{
    "question": "What is direct communication in message passing?",
    "answer": "In direct communication, processes explicitly name each other while sending and receiving messages."
},

{
    "question": "What is indirect communication?",
    "answer": "Indirect communication uses mailboxes or ports to exchange messages."
},

{
    "question": "What is a mailbox in IPC?",
    "answer": "A mailbox is an object used to store messages exchanged between processes."
},

{
    "question": "What is synchronous message passing?",
    "answer": "Synchronous message passing blocks the sender or receiver until communication completes."
},

{
    "question": "What is asynchronous message passing?",
    "answer": "Asynchronous message passing allows processes to continue execution without waiting."
},

{
    "question": "What is buffering in message passing?",
    "answer": "Buffering refers to temporary storage of messages in a queue associated with a communication link."
},

{
    "question": "What is zero-capacity buffering?",
    "answer": "Zero-capacity buffering stores no messages and requires the sender to wait for the receiver."
},

{
    "question": "What is bounded-capacity buffering?",
    "answer": "Bounded-capacity buffering stores a finite number of messages."
},

{
    "question": "What is unbounded-capacity buffering?",
    "answer": "Unbounded-capacity buffering stores an unlimited number of messages and the sender never waits."
},

{
    "question": "What is a pipe?",
    "answer": "A pipe is one of the oldest IPC mechanisms in UNIX used for communication between related processes."
},

{
    "question": "What are the limitations of ordinary pipes?",
    "answer": "Ordinary pipes are half-duplex and can only be used between processes having a common ancestor."
},

{
    "question": "What is a named pipe?",
    "answer": "A named pipe is a pipe that allows communication between unrelated processes."
},

{
    "question": "What is a race condition?",
    "answer": "A race condition occurs when multiple processes access shared data simultaneously and the result depends on execution order."
},

{
    "question": "What is the critical section problem?",
    "answer": "The critical section problem involves ensuring that only one process executes in its critical section at a time."
},

{
    "question": "What are the three requirements for a critical section solution?",
    "answer": "Mutual exclusion, progress, and bounded waiting."
},

{
    "question": "Define mutual exclusion.",
    "answer": "Mutual exclusion ensures that only one process can execute in its critical section at a time."
},

{
    "question": "Define progress.",
    "answer": "Progress ensures that if no process is in its critical section, one of the waiting processes will eventually enter."
},

{
    "question": "Define bounded waiting.",
    "answer": "Bounded waiting guarantees that a process will not wait indefinitely before entering its critical section."
},

{
    "question": "What is Peterson's solution?",
    "answer": "Peterson's solution is a software-based solution to the critical section problem for two processes."
},

{
    "question": "What is a semaphore?",
    "answer": "A semaphore is a synchronization variable used to control access to shared resources."
},

{
    "question": "What are the operations associated with semaphores?",
    "answer": "The two operations are wait() and signal()."
},

{
    "question": "What does wait() do?",
    "answer": "wait() decrements the semaphore value and blocks the process if the value becomes negative."
},

{
    "question": "What does signal() do?",
    "answer": "signal() increments the semaphore value and wakes up blocked processes."
},

{
    "question": "What is a binary semaphore?",
    "answer": "A binary semaphore takes values 0 and 1 and is mainly used for mutual exclusion."
},

{
    "question": "What is a counting semaphore?",
    "answer": "A counting semaphore allows multiple instances of a resource to be managed."
},

{
    "question": "What is busy waiting?",
    "answer": "Busy waiting occurs when a process continuously checks a condition without relinquishing the CPU."
},

{
    "question": "What is a mutex lock?",
    "answer": "A mutex lock provides mutual exclusion by allowing only one thread to access a resource at a time."
},

{
    "question": "What is a monitor?",
    "answer": "A monitor is a high-level synchronization construct that encapsulates shared variables and procedures."
},

{
    "question": "What is a condition variable?",
    "answer": "A condition variable allows processes to wait and be signaled inside a monitor."
},

{
    "question": "What is deadlock?",
    "answer": "Deadlock is a situation in which a set of processes are permanently blocked because each process waits for resources held by others."
},

{
    "question": "What are the four necessary conditions for deadlock?",
    "answer": "Mutual exclusion, hold and wait, no preemption, and circular wait."
},

{
    "question": "What is mutual exclusion in deadlock?",
    "answer": "Only one process can use a resource at a time."
},

{
    "question": "What is hold and wait?",
    "answer": "Processes holding resources are waiting for additional resources."
},

{
    "question": "What is no preemption?",
    "answer": "Resources cannot be forcibly taken away from processes."
},

{
    "question": "What is circular wait?",
    "answer": "A set of processes exists such that each process waits for a resource held by the next process."
},

{
    "question": "What are the methods of handling deadlocks?",
    "answer": "Deadlock prevention, avoidance, detection, and recovery."
},

{
    "question": "What is deadlock prevention?",
    "answer": "Deadlock prevention ensures that at least one of the four necessary conditions never holds."
},

{
    "question": "What is deadlock avoidance?",
    "answer": "Deadlock avoidance dynamically allocates resources to ensure the system remains in a safe state."
},

{
    "question": "What is a safe state?",
    "answer": "A safe state is a state in which processes can complete execution without causing deadlock."
},

{
    "question": "What is the Banker's Algorithm?",
    "answer": "Banker's Algorithm is a deadlock avoidance algorithm that allocates resources only if the resulting state remains safe."
}

])

In [8]:
# Batch 3 (Q101–Q150): Memory Management and Virtual Memory

raw_data.extend([

{
    "question": "What is memory management?",
    "answer": "Memory management is the functionality of an operating system responsible for managing primary memory and allocating memory to processes efficiently."
},

{
    "question": "Why is memory protection required?",
    "answer": "Memory protection prevents processes from accessing memory belonging to other processes or the operating system."
},

{
    "question": "What are base and limit registers?",
    "answer": "Base and limit registers are hardware registers used for memory protection. The base register stores the starting address, while the limit register stores the size of the address space."
},

{
    "question": "What is address binding?",
    "answer": "Address binding is the process of mapping logical addresses generated by a program to physical addresses in memory."
},

{
    "question": "What are the three types of address binding?",
    "answer": "The three types are compile-time binding, load-time binding, and execution-time binding."
},

{
    "question": "What is compile-time binding?",
    "answer": "Compile-time binding generates absolute addresses if the location of the process in memory is known beforehand."
},

{
    "question": "What is load-time binding?",
    "answer": "Load-time binding generates relocatable code and binds addresses when the program is loaded into memory."
},

{
    "question": "What is execution-time binding?",
    "answer": "Execution-time binding maps logical addresses to physical addresses during execution using hardware support."
},

{
    "question": "What is a logical address?",
    "answer": "A logical address is the address generated by the CPU during program execution."
},

{
    "question": "What is a physical address?",
    "answer": "A physical address is the actual address in main memory where data resides."
},

{
    "question": "What is an MMU?",
    "answer": "The Memory Management Unit (MMU) is hardware that maps logical addresses to physical addresses."
},

{
    "question": "What is dynamic relocation?",
    "answer": "Dynamic relocation is the process of translating logical addresses into physical addresses during execution."
},

{
    "question": "What is swapping?",
    "answer": "Swapping is the process of moving processes between main memory and secondary storage to free memory space."
},

{
    "question": "What is contiguous memory allocation?",
    "answer": "Contiguous memory allocation assigns each process a single continuous block of memory."
},

{
    "question": "What is fragmentation?",
    "answer": "Fragmentation refers to wasted memory space caused by inefficient allocation."
},

{
    "question": "What is internal fragmentation?",
    "answer": "Internal fragmentation occurs when allocated memory blocks are larger than required, wasting space inside the block."
},

{
    "question": "What is external fragmentation?",
    "answer": "External fragmentation occurs when free memory is scattered into small non-contiguous blocks."
},

{
    "question": "What is compaction?",
    "answer": "Compaction is the process of combining fragmented free spaces into one contiguous block."
},

{
    "question": "What is paging?",
    "answer": "Paging is a memory management scheme that divides logical memory into fixed-size pages and physical memory into frames."
},

{
    "question": "What is a page?",
    "answer": "A page is a fixed-size block of logical memory."
},

{
    "question": "What is a frame?",
    "answer": "A frame is a fixed-size block of physical memory."
},

{
    "question": "What is a page table?",
    "answer": "A page table stores the mapping between pages and frames."
},

{
    "question": "Why is paging used?",
    "answer": "Paging eliminates external fragmentation and simplifies memory allocation."
},

{
    "question": "What is page-table base register (PTBR)?",
    "answer": "PTBR stores the address of the page table for the currently executing process."
},

{
    "question": "What is a TLB?",
    "answer": "A Translation Lookaside Buffer (TLB) is a cache that stores recently used page table entries."
},

{
    "question": "Why is TLB used?",
    "answer": "TLB reduces memory access time by avoiding repeated page table lookups."
},

{
    "question": "What is TLB hit?",
    "answer": "A TLB hit occurs when the required page entry is found in the TLB."
},

{
    "question": "What is TLB miss?",
    "answer": "A TLB miss occurs when the page entry is absent in the TLB, requiring access to the page table."
},

{
    "question": "What is effective memory access time?",
    "answer": "Effective memory access time is the average time required to access memory considering TLB hits and misses."
},

{
    "question": "What is segmentation?",
    "answer": "Segmentation divides memory into variable-sized segments based on logical divisions like code, stack, and data."
},

{
    "question": "What is a segment table?",
    "answer": "A segment table stores the base address and limit of each segment."
},

{
    "question": "Differentiate paging and segmentation.",
    "answer": "Paging uses fixed-size blocks and avoids external fragmentation, whereas segmentation uses variable-sized blocks and reflects the logical structure of programs."
},

{
    "question": "What is virtual memory?",
    "answer": "Virtual memory is a technique that allows processes to execute even if they are not completely loaded into physical memory."
},

{
    "question": "What is demand paging?",
    "answer": "Demand paging loads pages into memory only when they are required."
},

{
    "question": "What is a page fault?",
    "answer": "A page fault occurs when a required page is not present in main memory."
},

{
    "question": "What happens during page fault handling?",
    "answer": "The operating system locates the page on disk, loads it into memory, updates the page table, and restarts the interrupted instruction."
},

{
    "question": "What is pure demand paging?",
    "answer": "Pure demand paging loads a page into memory only when it is referenced for the first time."
},

{
    "question": "What is page replacement?",
    "answer": "Page replacement selects a page to remove from memory when a new page must be loaded and no free frame is available."
},

{
    "question": "What is FIFO page replacement?",
    "answer": "FIFO replaces the oldest page in memory first."
},

{
    "question": "What is Belady's anomaly?",
    "answer": "Belady's anomaly is the phenomenon where increasing the number of frames increases the number of page faults."
},

{
    "question": "Which page replacement algorithm is optimal?",
    "answer": "Optimal page replacement is theoretically optimal because it replaces the page that will not be used for the longest time in the future."
},

{
    "question": "Why is the Optimal algorithm difficult to implement?",
    "answer": "Because it requires knowledge of future page references."
},

{
    "question": "What is LRU page replacement?",
    "answer": "Least Recently Used (LRU) replaces the page that has not been used for the longest time."
},

{
    "question": "Why is LRU considered better than FIFO?",
    "answer": "Because LRU approximates optimal replacement and does not suffer from Belady's anomaly."
},

{
    "question": "What is the Second-Chance algorithm?",
    "answer": "The Second-Chance algorithm is a modified FIFO algorithm that gives pages with reference bit 1 another chance."
},

{
    "question": "What is a reference bit?",
    "answer": "A reference bit indicates whether a page has been recently accessed."
},

{
    "question": "What is thrashing?",
    "answer": "Thrashing is a condition in which excessive page faults reduce CPU utilization."
},

{
    "question": "What causes thrashing?",
    "answer": "Thrashing occurs when processes do not have enough frames and spend most of their time swapping pages."
},

{
    "question": "What is the working-set model?",
    "answer": "The working-set model keeps track of pages actively used by a process to reduce thrashing."
},

{
    "question": "What is locality of reference?",
    "answer": "Locality of reference is the tendency of programs to access the same set of memory locations repeatedly."
},

{
    "question": "What are the two types of locality of reference?",
    "answer": "Temporal locality and spatial locality."
}

])

In [9]:
# Batch 4 (Q151–Q200): File Systems, Disk Scheduling and Storage Management

raw_data.extend([

{
    "question": "What is a file?",
    "answer": "A file is a named collection of related information stored on secondary storage."
},

{
    "question": "What is a file system?",
    "answer": "A file system is a method used by the operating system to organize, store, retrieve, and manage files on storage devices."
},

{
    "question": "What is a directory?",
    "answer": "A directory is a collection of information about files and subdirectories."
},

{
    "question": "What are the operations performed on a directory?",
    "answer": "Searching, creating files, deleting files, listing directories, renaming files, and traversing the file system."
},

{
    "question": "What is a path name?",
    "answer": "A path name specifies the location of a file within the directory hierarchy."
},

{
    "question": "What is an absolute path?",
    "answer": "An absolute path specifies the complete path from the root directory to a file."
},

{
    "question": "What is a relative path?",
    "answer": "A relative path specifies the path with respect to the current working directory."
},

{
    "question": "What is a single-level directory structure?",
    "answer": "A single-level directory structure contains one directory for all users, leading to naming conflicts."
},

{
    "question": "What are the disadvantages of a single-level directory?",
    "answer": "Naming problems and lack of grouping capability."
},

{
    "question": "What is a two-level directory structure?",
    "answer": "In a two-level directory, each user has a separate directory."
},

{
    "question": "What is a tree-structured directory?",
    "answer": "A tree-structured directory organizes files hierarchically with a root directory and subdirectories."
},

{
    "question": "What is an acyclic graph directory?",
    "answer": "An acyclic graph directory allows sharing of files and subdirectories without creating cycles."
},

{
    "question": "What is aliasing in directories?",
    "answer": "Aliasing refers to having multiple names for the same file."
},

{
    "question": "What is a dangling pointer in file systems?",
    "answer": "A dangling pointer refers to a link pointing to a deleted file."
},

{
    "question": "What is a general graph directory?",
    "answer": "A general graph directory permits cycles and multiple parent directories."
},

{
    "question": "Why is garbage collection required in general graph directories?",
    "answer": "Garbage collection removes inaccessible files and prevents wasted storage."
},

{
    "question": "What is file allocation?",
    "answer": "File allocation determines how disk blocks are assigned to files."
},

{
    "question": "What are the three file allocation methods?",
    "answer": "Contiguous allocation, linked allocation, and indexed allocation."
},

{
    "question": "What is contiguous allocation?",
    "answer": "Contiguous allocation stores all file blocks consecutively on disk."
},

{
    "question": "What are the advantages of contiguous allocation?",
    "answer": "Fast sequential access and efficient direct access."
},

{
    "question": "What are the disadvantages of contiguous allocation?",
    "answer": "External fragmentation and difficulty in file expansion."
},

{
    "question": "What is linked allocation?",
    "answer": "Linked allocation stores file blocks as linked lists distributed across the disk."
},

{
    "question": "What is the main advantage of linked allocation?",
    "answer": "No external fragmentation."
},

{
    "question": "What is the drawback of linked allocation?",
    "answer": "Poor random access performance."
},

{
    "question": "What is indexed allocation?",
    "answer": "Indexed allocation uses an index block containing pointers to all file blocks."
},

{
    "question": "What is the advantage of indexed allocation?",
    "answer": "Supports direct access and avoids external fragmentation."
},

{
    "question": "What is free-space management?",
    "answer": "Free-space management keeps track of available disk blocks."
},

{
    "question": "What are the methods of free-space management?",
    "answer": "Bit vector, linked list, grouping, and counting."
},

{
    "question": "What is a bit vector?",
    "answer": "A bit vector uses bits to represent allocated and free disk blocks."
},

{
    "question": "What is disk scheduling?",
    "answer": "Disk scheduling determines the order in which disk I/O requests are serviced."
},

{
    "question": "Why is disk scheduling required?",
    "answer": "To minimize seek time and improve disk throughput."
},

{
    "question": "What is seek time?",
    "answer": "Seek time is the time required to move the disk head to the desired track."
},

{
    "question": "What is rotational latency?",
    "answer": "Rotational latency is the time waiting for the desired sector to rotate under the disk head."
},

{
    "question": "What is transfer time?",
    "answer": "Transfer time is the time needed to transfer data between disk and memory."
},

{
    "question": "What is FCFS disk scheduling?",
    "answer": "FCFS services requests in the order they arrive."
},

{
    "question": "What is the disadvantage of FCFS disk scheduling?",
    "answer": "It may result in large seek times and poor performance."
},

{
    "question": "What is SSTF disk scheduling?",
    "answer": "Shortest Seek Time First services the request closest to the current head position."
},

{
    "question": "What is the drawback of SSTF?",
    "answer": "It may cause starvation of requests far from the head."
},

{
    "question": "What is SCAN disk scheduling?",
    "answer": "SCAN moves the disk arm in one direction servicing requests and then reverses direction."
},

{
    "question": "Why is SCAN called the elevator algorithm?",
    "answer": "Because the disk head moves back and forth similar to an elevator."
},

{
    "question": "What is C-SCAN scheduling?",
    "answer": "Circular SCAN services requests in one direction and returns to the beginning without servicing requests during the return."
},

{
    "question": "What is LOOK scheduling?",
    "answer": "LOOK reverses direction only after reaching the last request instead of the end of the disk."
},

{
    "question": "What is C-LOOK scheduling?",
    "answer": "C-LOOK services requests in one direction and jumps to the lowest pending request."
},

{
    "question": "Which disk scheduling algorithm provides better uniform waiting time?",
    "answer": "C-SCAN provides more uniform waiting time."
},

{
    "question": "What is RAID?",
    "answer": "RAID stands for Redundant Array of Independent Disks and is used to improve performance and reliability."
},

{
    "question": "What is RAID 0?",
    "answer": "RAID 0 uses striping to improve performance but provides no fault tolerance."
},

{
    "question": "What is RAID 1?",
    "answer": "RAID 1 uses mirroring to provide redundancy."
},

{
    "question": "What is RAID 5?",
    "answer": "RAID 5 uses block-level striping with distributed parity for fault tolerance."
},

{
    "question": "What is a partition?",
    "answer": "A partition is a logical division of a disk."
},

{
    "question": "What is a volume?",
    "answer": "A volume is a storage area containing a file system."
},

{
    "question": "What is formatting?",
    "answer": "Formatting prepares a disk for storing files by creating a file system."
}

])

In [10]:
dataset = Dataset.from_list(raw_data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Train: {len(dataset['train'])} examples")
print(f"Test: {len(dataset['test'])} examples")
print("\nSample entry:")
print(dataset['train'][0])

Train: 183 examples
Test: 21 examples

Sample entry:
{'question': 'What is a volume?', 'answer': 'A volume is a storage area containing a file system.'}


In [11]:
# Cell 3: Format dataset using Phi-3's chat template
def format_prompt(example):
    """
    Phi-3 uses a specific chat format:
    <|user|>\n{question}<|end|>\n<|assistant|>\n{answer}<|end|>
    """
    if "question" in example:
        # Custom dataset
        prompt = f"<|user|>\n{example['question']}<|end|>\n<|assistant|>\n{example['answer']}<|end|>"
    else:
        # APPS dataset — extract first solution
        solutions = json.loads(example.get("solutions", "[]"))
        if not solutions:
            return {"text": ""}
        answer = solutions[0]
        prompt = f"<|user|>\nSolve this programming problem:\n\n{example['question']}<|end|>\n<|assistant|>\n```python\n{answer}\n```<|end|>"

    return {"text": prompt}

# Apply formatting
formatted = dataset["train"].map(format_prompt, remove_columns=dataset["train"].column_names)
formatted = formatted.filter(lambda x: len(x["text"]) > 50)  # remove empty
print(formatted[0]["text"][:500])

Map:   0%|          | 0/183 [00:00<?, ? examples/s]

Filter:   0%|          | 0/183 [00:00<?, ? examples/s]

<|user|>
What is a volume?<|end|>
<|assistant|>
A volume is a storage area containing a file system.<|end|>


In [12]:
# Cell 4: Load model with 4-bit quantization (fits in Colab T4's 15GB VRAM)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,      # nested quantization saves more memory
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best for LLM weights
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # important for causal LMs

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",            # auto-distributes across available GPUs/CPU
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"   # use "flash_attention_2" if supported
)

print(f"Model loaded. Parameters: {model.num_parameters():,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded. Parameters: 3,821,079,552


In [13]:
# Cell 5: Attach LoRA adapters — only ~0.23% of parameters are trained
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                    # rank — higher = more capacity, more memory
    lora_alpha=32,           # scaling factor (usually 2x rank)
    target_modules=[         # which weight matrices to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj"         # MLP/FFN
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Output: trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327
# Note: Phi-3 fuses q/k/v into qkv_proj and gate/up into gate_up_proj, so of the
# names above only o_proj and down_proj exist. Those two layers get the adapters.

trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327


In [14]:
# Cell 6: Fine-tune using HuggingFace TRL's SFTTrainer
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

OUTPUT_DIR = "./phi3-os-finetuned"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,              # 3 passes over the dataset
    per_device_train_batch_size=2,   # keep low for T4
    gradient_accumulation_steps=8,   # effective batch = 2×8 = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=False,
    bf16=True,                       # bfloat16 — better than fp16 for training
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",               # remove if not using W&B
    run_name="phi3-os-qlora",
    optim="paged_adamw_8bit",        # memory-efficient optimizer from bitsandbytes
    max_grad_norm=0.3,
    group_by_length=True,            # batch similar-length sequences → less padding
)

# Only compute loss on the assistant's response, not the question
response_template = "<|assistant|>"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted,
    eval_dataset=formatted.select(range(50)),  # small eval split
    tokenizer=tokenizer,
    data_collator=collator,
    dataset_text_field="text",
    max_seq_length=1024,
)

# 🚀 Start training
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training complete!")

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will over

Map:   0%|          | 0/183 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
0,1.641600,1.029897
2,0.908900,0.817062


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training complete!


In [ ]:
# Cell 7: Load and test the fine-tuned model
from peft import PeftModel

# Load base model + LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model_ft = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model_ft.eval()

def generate_response(question, max_new_tokens=512):
    prompt = f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model_ft.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (not the prompt)
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

# Test it
question = "What are the four necessary conditions for a deadlock?"
print(generate_response(question))

In [16]:
from huggingface_hub import login
login()

In [18]:
from peft import PeftModel
import torch

merged_model = model_ft.merge_and_unload()

HF_USERNAME = "Prithvi-Aithal"
REPO_NAME = "phi3-mini-os-finetuned"

merged_model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", private=False)
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
print(f"Model live at: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:336: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...86xzfv3/model.safetensors:   2%|2         | 56.3MB / 2.26GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pnhwi07j2/tokenizer.model:  75%|#######4  |  374kB /  500kB            

Model live at: https://huggingface.co/Prithvi-Aithal/phi3-mini-os-finetuned
